# Train


In [ ]:
#| default_exp train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn as nn, lightning.pytorch as pl, warnings

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts

from physiojepa.transformers import PatchTSJEPA
from physiojepa.loss import FocalLoss

In [ ]:
#| export
class PatchTSJEPALightning(pl.LightningModule):
    def __init__(self,
                 learning_rate,
                 train_size,
                 batch_size,
                 channels,
                 patchtsjepa_encoder_kwargs,
                 patchtsjepa_predictor_kwargs,
                 use_sequence_padding_mask=False,
                 loss_func = 'smoothl1',
                 max_lr=0.01,
                 weight_decay=0.,
                 epochs=100,
                 optimizer_type='adamw',
                 scheduler_type='OneCycle',
                 target_mask_range=(0.1,0.5), # the target can be up to 50% of the original x 
                 context_mask_range=(0.2,0.8), # the context can be up to 80% of masked out target (1-target_mask_ratio)
                 mask_input_ratio=0.0,
                 mask_all_channels=True,
                 pretrain=True,
                 ema_decay=0.996,
                 ):
        super().__init__()
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.max_lr = max_lr
        self.train_size = train_size
        self.batch_size = batch_size
        self.epochs = epochs
        self.channels = channels
        self.use_sequence_padding_mask = use_sequence_padding_mask
        self.optimizer_type = optimizer_type
        self.weight_decay = weight_decay
        self.loss_fn = nn.SmoothL1Loss() if loss_func == 'smoothl1' else nn.MSELoss()
        self.target_mask_range = target_mask_range
        self.context_mask_range = context_mask_range
        self.pretrain = pretrain
        self.mask_all_channels = mask_all_channels
        self.ema_decay = ema_decay
        self.mask_input_ratio = mask_input_ratio
        ipe = self.train_size//self.batch_size
        self.momentum_scheduler = (self.ema_decay + i*(1-self.ema_decay)/(ipe*self.epochs) for i in range(int(ipe*self.epochs)+1))
        self.model = PatchTSJEPA(patchtsjepa_encoder_kwargs, patchtsjepa_predictor_kwargs, pretrain=pretrain, target_mask_range=self.target_mask_range, context_mask_range=self.context_mask_range, mask_all_channels=self.mask_all_channels, mask_input_ratio=self.mask_input_ratio)

    def ema_update(self, context_encoder, target_encoder):
        with torch.no_grad():
            m = next(self.momentum_scheduler)
            for param_q, param_k in zip(context_encoder.parameters(), target_encoder.parameters()):
                param_k.data.mul_(m).add_((1.-m) * param_q.detach().data)
                param_k.requires_grad_(False)
        return target_encoder

    def forward(self, x, sequence_padding_mask=None):
        if self.pretrain or self.training:
            pred, z_target, z_context, context_mask, target_mask, padding_mask = self.model(x, sequence_padding_mask=sequence_padding_mask)
            return pred, z_target, z_context, context_mask, target_mask, padding_mask
        else:
            z, padding_mask = self.model(x, sequence_padding_mask=sequence_padding_mask)
            return z, padding_mask

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.use_sequence_padding_mask:
            x, _, sequence_padding_mask = batch
        else:
            x, _ = batch
            sequence_padding_mask = None
        pred, z_target, z_context, context_mask, target_mask, padding_mask = self(x, sequence_padding_mask=sequence_padding_mask)
        
        loss = self.loss_fn(pred, z_target)

        loss = loss.to(self.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss
    
    def on_train_batch_end(self, outputs, batch, batch_idx):
        """Called after each training batch ends"""
        # Update target encoder weights using EMA
        self.model.target_encoder = self.ema_update(
            self.model.context_encoder, 
            self.model.target_encoder
        )
    
    def validation_step(self, batch, batch_idx):
        if self.use_sequence_padding_mask:
            x, _, sequence_padding_mask = batch
        else:
            x, _ = batch
            sequence_padding_mask = None
        pred, z_target, z_context, _, _, _ = self(x, sequence_padding_mask=sequence_padding_mask)

        loss = self.loss_fn(pred, z_target)
        loss = loss.to(self.device)

        self.log("val_loss", loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(optimizer, max_lr=self.max_lr, epochs=self.epochs, steps_per_epoch=(self.train_size//self.batch_size))
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=self.epochs//10, T_mult=2, eta_min=1e-8) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return optimizer

In [ ]:
#| export
class PatchTFTSingleOutcomeLightning(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                linear_probing_head, # model head to linear probe/train
                metrics={}, # name:function for metrics to log
                drop_channel_indexes=None, # indices of channels to drop from the model after encoding
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                loss_fxn='CrossEntropy', # loss function to use, BCEWithLogitsLoss, BCELos, or FocalLoss for binary classification
                class_weights=None, # weights of classes to use in CE loss fxn
                gamma=2., # for focal loss
                label_smoothing=0, # label smoothing for cross entropy loss
                use_sequence_padding_mask=False, #indicator to use the sequence padding mask when training/in the loss fxn
                y_padding_mask=-100, # padded value that was added to target and indice to ignore when computing loss
                max_lr=0.01, # maximum learning rate for one_cycle_scheduler
                epochs=100, # number of epochs for one_cycle_scheduler
                one_cycle_scheduler=True, # indicator to use a one cycle scheduler to vary the learning rate 
                scheduler_type='OneCycle',
                optimizer_type='Adam',
                weight_decay=0., # weight decay for Adam optimizer
                pretrained_encoder_path=None, # path of the pretrained model to use for linear probing
                preloaded_model=None, # loaded pretrained model to use for linear probing
                torch_model_name='model', # name of the pytorch model within the lightning model module, this is to remove layers (for example lightning_model.pytorch_model.head = nn.Identity())
                remove_pretrain_layers=['head'] # layers within the lightning model or lightning model.pytorch_model to remove
                ):
        super().__init__()
        self.encoder = preloaded_model
        if remove_pretrain_layers is not None and len(remove_pretrain_layers) > 0:
            for l in remove_pretrain_layers:
                if torch_model_name is not None:
                    setattr(getattr(self.encoder, torch_model_name), l, nn.Identity())
                else:
                    setattr(self.encoder, l, nn.Identity())
        #if bool(self.encoder.hparams.get('convolve')):
        #    assert self.encoder.hparams['conv_out_channels'] == linear_probing_kwargs['c_in'], f"The encoder used a convolution, creating {self.encoder.hparams['conv_out_channels']} channels. This should be the value for `c_in` in the time distributed feed forward network."
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.one_cycle_scheduler = one_cycle_scheduler
        self.weight_decay = weight_decay
        self.label_smoothing = label_smoothing
        self.learning_rate = learning_rate
        self.max_lr = max_lr
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size
        self.metrics = metrics
        self.y_padding_mask = y_padding_mask
        self.class_weights = class_weights
        self.use_sequence_padding_mask = use_sequence_padding_mask
        self.loss_fxn = loss_fxn.lower()
        self.gamma = gamma
        self.fine_tune = fine_tune
        self.optimizer_type = optimizer_type
        self.drop_channel_indexes = drop_channel_indexes
        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
        # Adjust the output layer for binary classification
        self.feedforward = linear_probing_head # model head to linear probe/train
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x, sequence_padding_mask=None):
        x = self.encoder(x, sequence_padding_mask=sequence_padding_mask) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if self.drop_channel_indexes is not None:
            channels_to_keep = [i for i in range(x.shape[1]) if i not in self.drop_channel_indexes]
            x = torch.index_select(x, dim=1, index=torch.tensor(channels_to_keep, device=x.device))
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        x = self.feedforward(x, return_softmax=True) # [bs, n_classes, pred_len_seconds]
        return x # Ensure the output is of shape [batch_size]
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0): #does the order matter? should it be after test_step()
        if self.use_sequence_padding_mask:
            x, _, sequence_padding_mask = batch
        else:
            x,_ = batch
            sequence_padding_mask = None
        x = self.encoder(x, sequence_padding_mask = sequence_padding_mask)
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if self.drop_channel_indexes is not None:
            channels_to_keep = [i for i in range(x.shape[1]) if i not in self.drop_channel_indexes]
            x = torch.index_select(x, dim=1, index=torch.tensor(channels_to_keep, device=x.device))
        preds = self.feedforward(x, return_softmax=True)
        return preds

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.use_sequence_padding_mask:
            x, y, sequence_padding_mask = batch
        else:
            x, y = batch
            sequence_padding_mask = None
        x = self.encoder(x, sequence_padding_mask=sequence_padding_mask)
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if self.drop_channel_indexes is not None:
            channels_to_keep = [i for i in range(x.shape[1]) if i not in self.drop_channel_indexes]
            x = torch.index_select(x, dim=1, index=torch.tensor(channels_to_keep, device=x.device))
        x = self.feedforward(x)  # Ensure output is [batch_size]
        bce_loss = nn.CrossEntropyLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, label_smoothing=self.label_smoothing, ignore_index=self.y_padding_mask)
        focal_loss = FocalLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, gamma=self.gamma, ignore_index=self.y_padding_mask)
        bce_loss_val = bce_loss(x,y)
        focal_loss_val = focal_loss(x,y)
        if bce_loss_val.isnan() and torch.all(y == self.y_padding_mask):
            bce_loss_val = torch.nan_to_num(bce_loss_val) # convert to 0
        if focal_loss_val.isnan() and torch.all(y == self.y_padding_mask):
            focal_loss_val = torch.nan_to_num(focal_loss_val) # convert to 0
        loss = bce_loss_val if self.loss_fxn == 'crossentropy' else focal_loss_val
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_ce_loss' if self.loss_fxn != 'crossentropy' else 'train_focal_loss', bce_loss_val if self.loss_fxn != 'crossentropy' else focal_loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        if x.dim() == 3:
            x = x[:,1,:].squeeze(-1)
        if y.dim() == 2:
            y = y.squeeze(-1)
        self.log_dict({f'train_{name}':metric(x, y) for name, metric in self.metrics.items()}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        if self.use_sequence_padding_mask:
            x, y, sequence_padding_mask = batch
        else:
            x, y = batch
            sequence_padding_mask = None
        x = self.encoder(x, sequence_padding_mask=sequence_padding_mask)
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if self.drop_channel_indexes is not None:
            channels_to_keep = [i for i in range(x.shape[1]) if i not in self.drop_channel_indexes]
            x = torch.index_select(x, dim=1, index=torch.tensor(channels_to_keep, device=x.device))
        x = self.feedforward(x)  # Ensure output is [batch_size]
        bce_loss = nn.CrossEntropyLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, label_smoothing=self.label_smoothing, ignore_index=self.y_padding_mask)
        focal_loss =  FocalLoss(weight=self.class_weights.to(x.device) if self.class_weights is not None else None, gamma=self.gamma, ignore_index=self.y_padding_mask)
        bce_loss_val = bce_loss(x,y)
        focal_loss_val = focal_loss(x,y)
        if bce_loss_val.isnan() and torch.all(y == self.y_padding_mask):
            bce_loss_val = torch.nan_to_num(bce_loss_val) # convert to 0
        if focal_loss_val.isnan() and torch.all(y == self.y_padding_mask):
            focal_loss_val = torch.nan_to_num(focal_loss_val) # convert to 0
        loss = bce_loss_val if self.loss_fxn == 'crossentropy' else focal_loss_val 
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_ce_loss' if self.loss_fxn != 'crossentropy' else 'val_focal_loss', bce_loss_val if self.loss_fxn != 'crossentropy' else focal_loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        if x.dim() == 3:
            x = x[:,1,:].squeeze(-1)
        if y.dim() == 2:
            y = y.squeeze(-1)
        self.log_dict({f'val_{name}':metric(x, y) for name, metric in self.metrics.items()}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
    
    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(optimizer, max_lr=self.max_lr, epochs=self.epochs, steps_per_epoch=(self.train_size//self.batch_size))
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=self.epochs//10, T_mult=2, eta_min=1e-8)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()